In [ ]:
!python --version

In [ ]:
import os, sys
import pandas as pd
import requests
import time

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


### Defaults

In [ ]:
ROOT_DATA = '../data'
ROOT_DATA = Path(ROOT_DATA)

gdc = GDC(ROOT_DATA0=ROOT_DATA)

os.listdir(ROOT_DATA)[:10]


In [ ]:
os.listdir(gdc.root_data)[:10]

### Get all programs

In [ ]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)


### Primary sites given a program

In [ ]:
gdc.url_gdc_project

In [ ]:
prog_id = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

### GTEx

In [ ]:
root_gtex = ROOT_DATA / Path('GTEx')

os.listdir(root_gtex)[:10]


In [ ]:
fname_gtex = 'tcga_primary_site_to_gtex_ids.tsv'

dfg = pdreadcsv(fname_gtex, root_gtex)
print(len(dfg))

dfg.head(6)

In [ ]:
psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-LUAD'

dfa = dfg[dfg.tcga_project_id == psi_id]

gtex_id = None

if len(dfa) == 1:
    row = dfa.iloc[0]

    gtex_id = row.preferred_gtex_id
    gtex_tissue_ids = row.gtex_tissue_site_detail_ids
    print(f"Found '{gtex_id}' tissue '{gtex_tissue_ids}'")
elif len(dfa) == 0:
    print("Not found")
else:
    print("Multiple matches found")
    for i, row in dfa.iterrows():
        gtex_id = row.preferred_gtex_id
        gtex_tissue_ids = row.gtex_tissue_site_detail_ids

        print(f"{row.tcga_project_id} -> '{gtex_id}' tissue '{gtex_tissue_ids}'")


In [ ]:
gtex_id

### Get df_tumor

In [ ]:
psi_id

In [ ]:
verbose=False

dic_tumor, dic_normal = gdc.get_file_expression_tumor_and_normal(psi_id=psi_id, force=False, verbose=verbose)
df_tumor, df_normal = gdc.merge_normal_tumor_tables(dic_tumor, dic_normal,  imax_tumor=12, imax_normal=12, verbose=verbose)

len(df_tumor), len(df_normal)

In [ ]:
df_tumor.head(3)

In [ ]:
from scipy.stats import f_oneway

def add_entropy(df, read_limit:int=50, min_read:int=200, n_quantiles:int=10):
    # select tumor columns
    cols = [c for c in df.columns if c.startswith("tumor_")]

    def row_entropy(values):
        values = np.asarray(values, dtype=float)

        # remove NaNs
        values = values[np.isfinite(values)]

        if len(values) == 0:
            return np.nan

        # all equal → entropy = 0
        if np.allclose(values, values[0]):
            return 0.0

        # shift to positive (important if negative values exist)
        values = values - values.min()

        # avoid all zeros
        if np.allclose(values.sum(), 0):
            return 0.0

        probs = values / values.sum()

        # avoid log(0)
        probs = probs[probs > 0]

        entropy = -np.sum(probs * np.log2(probs))
        return entropy

    min_cols = int(len(cols) * 0.4)
    good = (df[cols] < read_limit).sum(axis=1) <= min_cols
    df = df[good].copy()

    df["total_sum"] = df[cols].sum(axis=1)

    df = df[df.total_sum > len(cols)*min_read]

    df["entropy"] = df[cols].apply(lambda row: row_entropy(row.values), axis=1)

    df["entropy_q"] = pd.qcut(df["entropy"], q=n_quantiles, labels=False, duplicates="drop" )

    return df


print(len(df_tumor))
df_tumor2 = add_entropy(df_tumor)
print(len(df_tumor2))
df_tumor2.head(3)


In [ ]:
def select_top_entropy(df, q=0.1):

    df = df[df.entropy_q <= q].copy()
    df = df.sort_values('entropy', ascending=True)
    df.reset_index(drop=True, inplace=True)

    return df

tumor_cols = [c for c in df_tumor.columns if c.startswith("tumor_")]

df_tumor3 = select_top_entropy(df_tumor2, q=0.1)
print(len(df_tumor3))
df_tumor3.head(3)

In [ ]:
df_tumor3.head(30)

In [ ]:
df_tumor3.tail(30)

In [ ]:
geneid_list = np.unique(df_tumor3['gene_id'])

len(geneid_list)

### GTEx portal

In [ ]:
print(len(geneid_list))
np.array(geneid_list[:50])

In [ ]:
GTEX_API = "https://gtexportal.org/api/v2"

In [ ]:
def resolve_gtex_gencode_id(gene_id, datasetId="gtex_v8", timeout=60):
    gene_base = str(gene_id).split(".")[0]

    url = f"{GTEX_API}/reference/gene"

    params = {
        "geneId": gene_base,
        "datasetId": datasetId,
    }

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    data = r.json().get("data", [])

    if not data:
        return None

    return data[0].get("gencodeId")

In [ ]:


def get_gtex_expression_for_geneid_list(
    gtex_id:str,
    geneid_list:list,
    datasetId:str="gtex_v8",
    page_size:int=1000,
    sleep: float = 0.1,
    timeout:int=60,
    verbose:bool=False,
):
    """
    Download GTEx normal tissue expression for selected genes.
    Returns long-format dataframe:
    geneSymbol | gencodeId | tissueSiteDetailId | sampleId | tpm | log2Tpm
    """

    all_rows = []

    print(">>>", len(geneid_list))
    for i, geneid in enumerate(geneid_list):
        page = 0

        while True:

            gencode_id = resolve_gtex_gencode_id(geneid)
            print(">>>", gencode_id, end=" ")

            if gencode_id is None:
                print("?")
                if verbose: print(f"Nothing found for {geneid}")
                break
            
            url = f"{GTEX_API}/expression/geneExpression"

            params = {
                "datasetId": datasetId,
                "gencodeId": gencode_id,
                "tissueSiteDetailId": gtex_id,
                "page": page,
                "itemsPerPage": page_size,
            }

            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()

            js = r.json()
            data = js.get("data", [])

            if not data:
                print("x")
                if verbose: print(f"Error: could not retrieve data for {gtex_id} for geneid '{geneid}'")
                break

            print("Ok")

            all_rows.extend(data)

            paging = js.get("paging_info", {})
            n_pages = paging.get("numberOfPages", 1)

            page += 1
            if page >= n_pages:
                break

            time.sleep(sleep)

    print("")
    df = pd.DataFrame(all_rows)
    return df

In [ ]:
df_gtex = get_gtex_expression_for_geneid_list(
                gtex_id=gtex_id,
                geneid_list=geneid_list,
                datasetId="gtex_v8",
                page_size=1000,
                sleep = 0.1,
                timeout=60)

print(df_gtex.shape)

In [ ]:
gdc.root_gtex = create_dir(gdc.ROOT_DATA0, 'GTEx')

In [ ]:

fname = "gtex_TPM_%s.tsv"%(gtex_id)
pdwritecsv(df_gtex, fname, gdc.root_gtex)

print(len(df_gtex))
df_gtex.head()

### GTEx download

https://www.gtexportal.org/home/downloads/adult-gtex/overview

In [ ]:
def download_file(url, filename_out):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        try:
            with open(filename_out, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        except Exception as e:
            print(f"Error: as writingfile {filename_out}: {e}")



In [ ]:
url_counts = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
)

url_tpm = (
    "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq/"
    "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct.gz"
)

url_meta = (
    "https://storage.googleapis.com/adult-gtex/metadata/"
    "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
)
fname_counts = "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_reads.gct.gz"
fname_meta   = "GTEx_Analysis_v11_Annotations_SampleAttributesDS.tsv"
fname_pheno = "GTEx_Analysis_v11_Annotations_SubjectPhenotypesDS.tsv"

filename_counts = gdc.root_gtex / fname_counts
filename_meta   = gdc.root_gtex / fname_meta
filename_pheno = gdc.root_gtex / fname_pheno

# --> download manually at https://www.gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
#                      and https://www.gtexportal.org/home/downloads/adult-gtex/metadata

# download_file(url_counts, filename_counts)
# download_file(url_meta, filename_meta)

### Reading GTEx count super-file

In [ ]:
# load matrix'1
dfcounts = pdreadcsv(fname_counts, gdc.root_gtex, skiprows=2)
print(len(dfcounts))
dfcounts.head(3)

### Phenotyp - DTHHRDY (Hardy Scale of Death)

> It indicates how the donor died and how fast, which affects tissue quality.  


| Value	| Meaning  |
|---------|-----------|
| 0	| Ventilator case (on life support before death) |
| 1	| Violent and fast death (<10 min) |
| 2	| Fast death (10 min – 1 hour) |
| 3	| Intermediate death (1 – 24 hours) |
| 4	| Slow death (>24 hours, prolonged illness) |
| NA	| Unknown |


In [ ]:
# load metadata
df_pheno = pdreadcsv(fname_pheno, gdc.root_gtex)
# df_pheno['DTHHRDY'] = df_pheno.DTHHRDY.astype(int)

print(len(df_pheno))
df_pheno.head(3)

In [ ]:
# load metadata
dfmeta = pdreadcsv(fname_meta, gdc.root_gtex)
print(len(dfmeta))
dfmeta.head(3)

### Healthy donors

1. Filter the tissue
2. Filter for high quality (Hardy Scale 1 or 2 are 'fast' deaths, less stress)
  . DTHHRDY: 1 = Ventilator, 2 = Fast death of natural causes
3. Sort by RIN score (SMRIN) to get the best preserved RNA

In [ ]:
# 1. Filter for Lung
df_ctrl = dfmeta[dfmeta["SMTSD"] == "Lung"].copy()

df_ctrl["SUBJID"] = df_ctrl["SAMPID"].str.split("-").str[:2].str.join("-")
df_ctrl = df_ctrl.merge(df_pheno, on="SUBJID", how="left")

# 2. Filter for high quality (Hardy Scale 1 or 2 are 'fast' deaths, less stress)
# DTHHRDY: 1 = Ventilator, 2 = Fast death of natural causes
df_ctrl = df_ctrl[df_ctrl["DTHHRDY"].isin([1, 2])]

# 3. Sort by RIN score (SMRIN) to get the best preserved RNA
# Then take the top 15
df_ctrl = df_ctrl.sort_values("SMRIN", ascending=False)
df_ctrl.reset_index(drop=True, inplace=True)

print(df_ctrl.shape)
df_ctrl.head(5)

In [ ]:
# select lung samples
Nsamples=15

lung_samples = df_ctrl.head(Nsamples*3)["SAMPID"].to_list()
lung_samples[:3]

In [ ]:
cols = ["Name", "Description"] + list(lung_samples)

In [ ]:
good_cols = [x for x in cols if x in dfcounts.columns]

if len(good_cols) > Nsamples:
    good_cols = good_cols[:Nsamples]
    
len(good_cols)

In [ ]:
df2 = dfcounts.loc[:, good_cols].copy()

In [ ]:
df2.rename(columns={"Name": "ensemblid", "Description": "symbol"}, inplace=True)

print(df2.shape)
df2.head(3).T

In [ ]:
fname = "gtex_expresson_count_%s.tsv"%(gtex_id)
pdwritecsv(df2, fname, gdc.root_gtex)

